# Fake Reviews Dataset — Feature Engineering

Builds linguistic/stylistic features from the cleaned dataset produced by the **Clean & EDA** notebook, ready to feed into classical ML models.

**Features built** (all grounded in what the EDA already showed matters, plus standard fake/AI-text-detection literature signals):

| Group | Features |
|---|---|
| Length & structure | `word_count` (confirmed signal from EDA: fake reviews shorter), `char_count`, `avg_word_length`, `sentence_count`, `avg_sentence_length` |
| Readability | `flesch_reading_ease`, `flesch_kincaid_grade` (AI-generated text often scores differently — either unusually simple or unusually uniform) |
| Lexical diversity | `unique_word_ratio` (unique words / total words — repetition patterns differ between generated and human text) |
| Punctuation / style | `exclamation_ratio` (per 100 words), `capital_word_ratio` (ALL-CAPS word density — enthusiasm marker), `punctuation_density` |
| Sentiment (TextBlob) | `sentiment_polarity` (-1 to 1), `sentiment_subjectivity` (0 to 1) |
| POS-tag ratios (NLTK) | `adjective_ratio` (classic "salesy language" signal), `noun_ratio`, `verb_ratio` |
| Context (carried over, not engineered) | `rating`, `category`, `label_clean` |

**Note on category confound check:** this notebook also prints class balance *per category* so you can see whether fake/real is roughly balanced within every category, or concentrated in a few — important since category is a potential confound if left unchecked.

## Setup

In [2]:
import re
import string

import pandas as pd
import numpy as np

import nltk
from nltk import pos_tag, word_tokenize
from textblob import TextBlob
import textstat


In [3]:
# Ensure NLTK resources are present (no-op if already downloaded)
resource_paths = {
    "punkt": "tokenizers/punkt",
    "punkt_tab": "tokenizers/punkt_tab",
    "averaged_perceptron_tagger": "taggers/averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng": "taggers/averaged_perceptron_tagger_eng",
}
for resource, path in resource_paths.items():
    try:
        nltk.data.find(path)
    except LookupError:
        print(f"Downloading missing NLTK resource: {resource}")
        nltk.download(resource, quiet=True)


## Configuration

In [4]:
INPUT_PATH = "data/processed/fake_reviews_cleaned.csv"     # cleaned dataset CSV (from the Clean & EDA notebook)
OUTPUT_PATH = "data/processed/fake_reviews_features.csv"   # where to save the feature table
SAMPLE = None  # optional: set an int to process only a random sample of N rows first (recommended for a quick test run before running on all 40k)


## Load Data

In [5]:
df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(df)} rows from {INPUT_PATH}")

df.head()


Loaded 40405 rows from data/processed/fake_reviews_cleaned.csv


,category,rating,label,review_text,label_clean,review_word_count
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfort...",fake,12
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I'...",fake,16
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...,fake,14
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i...",fake,17
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...,fake,18


## 1. Feature Extraction Functions

Length & structure, lexical diversity, and punctuation/style helpers.

In [6]:
def word_count(text: str) -> int:
    return len(text.split())


def char_count(text: str) -> int:
    return len(text)


def avg_word_length(text: str) -> float:
    words = text.split()
    if not words:
        return 0.0
    return np.mean([len(w) for w in words])


def sentence_count(text: str) -> int:
    # simple heuristic split; textstat has a more robust version we use for readability
    sentences = re.split(r"[.!?]+", text)
    return max(1, len([s for s in sentences if s.strip()]))


def unique_word_ratio(text: str) -> float:
    words = text.lower().split()
    if not words:
        return 0.0
    return len(set(words)) / len(words)


def exclamation_ratio(text: str) -> float:
    wc = word_count(text)
    if wc == 0:
        return 0.0
    return (text.count("!") / wc) * 100


def capital_word_ratio(text: str) -> float:
    words = text.split()
    if not words:
        return 0.0
    caps = [w for w in words if w.isupper() and len(w) > 1]
    return len(caps) / len(words)


def punctuation_density(text: str) -> float:
    if len(text) == 0:
        return 0.0
    punct_count = sum(1 for ch in text if ch in string.punctuation)
    return punct_count / len(text)


## 2. Sentiment & POS-tag Functions

Sentiment via TextBlob, POS-tag ratios via NLTK.

In [7]:
def get_sentiment(text: str):
    blob = TextBlob(text)
    return blob.sentiment.polarity, blob.sentiment.subjectivity


def get_pos_ratios(text: str):
    try:
        tokens = word_tokenize(text)
        if not tokens:
            return 0.0, 0.0, 0.0
        tags = pos_tag(tokens)
        n = len(tags)
        adj = sum(1 for _, t in tags if t.startswith("JJ")) / n
        noun = sum(1 for _, t in tags if t.startswith("NN")) / n
        verb = sum(1 for _, t in tags if t.startswith("VB")) / n
        return adj, noun, verb
    except Exception:
        return 0.0, 0.0, 0.0


## 3. Category Confound Check

Prints class balance per category — important since category is a potential confound if left unchecked.

In [8]:
def check_category_confound(df: pd.DataFrame) -> None:
    if "category" not in df.columns or "label_clean" not in df.columns:
        return
    print("\nClass balance per category (proportion fake):")
    ct = pd.crosstab(df["category"], df["label_clean"], normalize="index")
    print(ct.round(3), "\n")


check_category_confound(df)



Class balance per category (proportion fake):
label_clean                    fake   real
category                                  
Books_5                       0.500  0.500
Clothing_Shoes_and_Jewelry_5  0.500  0.500
Electronics_5                 0.500  0.500
Home_and_Kitchen_5            0.500  0.500
Kindle_Store_5                0.500  0.500
Movies_and_TV_5               0.500  0.500
Pet_Supplies_5                0.500  0.500
Sports_and_Outdoors_5         0.500  0.500
Tools_and_Home_Improvement_5  0.500  0.500
Toys_and_Games_5              0.499  0.501 



## 4. Build the Feature Table

Runs all extraction steps in sequence. The sentiment and POS-tagging steps are the slowest — expect this to take a few minutes on the full ~40k-row dataset.

In [9]:
def build_features(df: pd.DataFrame, text_col: str = "review_text") -> pd.DataFrame:
    feats = pd.DataFrame(index=df.index)

    print("Building length & structure features...")
    feats["word_count"] = df[text_col].apply(word_count)
    feats["char_count"] = df[text_col].apply(char_count)
    feats["avg_word_length"] = df[text_col].apply(avg_word_length)
    feats["sentence_count"] = df[text_col].apply(sentence_count)
    feats["avg_sentence_length"] = feats["word_count"] / feats["sentence_count"].replace(0, 1)

    print("Building readability features (textstat)...")
    feats["flesch_reading_ease"] = df[text_col].apply(
        lambda t: textstat.flesch_reading_ease(t) if len(t.split()) > 2 else np.nan
    )
    feats["flesch_kincaid_grade"] = df[text_col].apply(
        lambda t: textstat.flesch_kincaid_grade(t) if len(t.split()) > 2 else np.nan
    )

    print("Building lexical diversity features...")
    feats["unique_word_ratio"] = df[text_col].apply(unique_word_ratio)

    print("Building punctuation/style features...")
    feats["exclamation_ratio"] = df[text_col].apply(exclamation_ratio)
    feats["capital_word_ratio"] = df[text_col].apply(capital_word_ratio)
    feats["punctuation_density"] = df[text_col].apply(punctuation_density)

    print("Building sentiment features (TextBlob)... this may take a few minutes on 40k rows")
    sentiment_results = df[text_col].apply(get_sentiment)
    feats["sentiment_polarity"] = sentiment_results.apply(lambda x: x[0])
    feats["sentiment_subjectivity"] = sentiment_results.apply(lambda x: x[1])

    print("Building POS-tag ratio features (NLTK)... this is the slowest step, please be patient")
    pos_results = df[text_col].apply(get_pos_ratios)
    feats["adjective_ratio"] = pos_results.apply(lambda x: x[0])
    feats["noun_ratio"] = pos_results.apply(lambda x: x[1])
    feats["verb_ratio"] = pos_results.apply(lambda x: x[2])

    # Carry over useful context columns
    for col in ["rating", "category", "label_clean"]:
        if col in df.columns:
            feats[col] = df[col].values

    return feats


In [10]:
features = build_features(df)
features.head()


Building length & structure features...
Building readability features (textstat)...
Building lexical diversity features...
Building punctuation/style features...
Building sentiment features (TextBlob)... this may take a few minutes on 40k rows
Building POS-tag ratio features (NLTK)... this is the slowest step, please be patient


,word_count,char_count,avg_word_length,sentence_count,avg_sentence_length,flesch_reading_ease,flesch_kincaid_grade,unique_word_ratio,exclamation_ratio,capital_word_ratio,punctuation_density,sentiment_polarity,sentiment_subjectivity,adjective_ratio,noun_ratio,verb_ratio,rating,category,label_clean
0,12,73,5.166667,4,3.0,73.845000,4.450000,0.916667,16.666667,0.0,0.068493,0.473750,0.8,0.111111,0.055556,0.166667,5.0,Home_and_Kitchen_5,fake
1,16,79,4.000000,2,8.0,87.677500,3.017500,0.937500,0.000000,0.0,0.037975,0.558333,0.7,0.105263,0.210526,0.157895,5.0,Home_and_Kitchen_5,fake
2,14,67,3.857143,2,7.0,103.044286,0.625714,0.928571,0.000000,0.0,0.029851,0.250000,0.3,0.000000,0.312500,0.125000,5.0,Home_and_Kitchen_5,fake
3,17,80,3.764706,2,8.5,80.097647,6.310588,1.000000,5.882353,0.0,0.025000,0.400000,0.4,0.052632,0.157895,0.157895,1.0,Home_and_Kitchen_5,fake
4,18,85,3.777778,3,6.0,99.000000,1.686667,0.944444,0.000000,0.0,0.023529,0.740000,0.8,0.100000,0.150000,0.250000,5.0,Home_and_Kitchen_5,fake


## 5. Save & Summarize

In [11]:
features.to_csv(OUTPUT_PATH, index=False)
print(f"Feature table saved to: {OUTPUT_PATH}")
print(f"Shape: {features.shape}")


Feature table saved to: data/processed/fake_reviews_features.csv
Shape: (40405, 19)


In [12]:
print("Feature summary stats by label:")
numeric_cols = features.select_dtypes(include=[np.number]).columns
if "label_clean" in features.columns:
    display(features.groupby("label_clean")[numeric_cols].mean().T)


Feature summary stats by label:


label_clean,fake,real
word_count,61.330362,73.641405
char_count,303.469936,395.078803
avg_word_length,4.041297,4.356426
sentence_count,5.997276,5.298442
avg_sentence_length,10.439083,13.475519
flesch_reading_ease,86.756076,77.218001
flesch_kincaid_grade,3.848316,5.951892
unique_word_ratio,0.750051,0.825077
exclamation_ratio,0.661685,1.167407
capital_word_ratio,0.007050,0.010407
